In [40]:
#to be run on deafultinterpreter 3.12.8 in \documents\venvs\py312\Scripts\python.exe
import nidaqmx
import nidaqmx.system
import matplotlib.pyplot as plt
import numpy as np
import time

from nidaqmx.constants import AcquisitionType

system = nidaqmx.system.System.local()
for device in system.devices:
    print(device)

Device(name=Dev1)


In [ ]:
%matplotlib widget
from ctypes import POINTER, c_double

import numpy as np
import matplotlib.pyplot as plt
import nidaqmx
from nidaqmx.constants import AcquisitionType, Edge, TerminalConfiguration

def check_alternating_levels(levels):
    if not levels:
        print("levels is empty")
        return False

    expected = levels[0]
    for index, value in enumerate(levels[1:], start=2):
        expected = "LOW" if expected == "HIGH" else "HIGH"
        if value != expected:
            print(f"Wrong value at position {index}: expected {expected}, got {value}")
            return False

    print("levels alternate correctly")
    return True

DEVICE = "Dev1" # check the actual name in NI MAX
AI_CHANNEL = "ai0" # where the 500 Hz square wave is wired
CLOCK_TERMINAL = "/Dev1/PFI0" # where the 1000 Hz trigger is wired (note leading "/")
N_SAMPLES = 10_000 #can be whatever, it's not mandatory to be the same as the trigger rate of the delay generator
#as it's the nr of trigger signal it takes before stopping the acquisition
RATE = 20_000.0
THRESHOLD = 1 # volts; midpoint of your logic levels



with nidaqmx.Task() as task:
    task.ai_channels.add_ai_voltage_chan(f"{DEVICE}/{AI_CHANNEL}", terminal_config=TerminalConfiguration.DIFF, min_val=-10.0, max_val=10.0)
    #task.triggers.start_trigger.cfg_dig_edge_start_trig(trigger_source="/Dev1/PFI0", trigger_edge=Edge.RISING)
    # Use the external trigger edges AS the sample clock: one sample acquired on every rising edge on PFI0.
    task.timing.cfg_samp_clk_timing(rate=RATE, source=CLOCK_TERMINAL,
    active_edge=Edge.RISING,
    sample_mode=AcquisitionType.FINITE,
    samps_per_chan=N_SAMPLES,
    ) #the rate must match the trigger rate of the delay generator
    
    task.start()
    # Blocks until N_SAMPLES clock edges have arrived (or it times out)
    data = task.read(number_of_samples_per_channel=N_SAMPLES, timeout=nidaqmx.constants.WAIT_INFINITELY)
    print(task.in_stream.total_samp_per_chan_acquired)

levels = ["HIGH" if v > THRESHOLD else "LOW" for v in data]
print(levels)
print(data)
check_alternating_levels(levels)

meas_data = np.array(data)

plt.close('all')
plt.figure()
plt.plot(meas_data, marker = '+', linestyle ='None')
#plt.plot(meas_data.flatten()) #actually flatten is not necessary, We already have a 1d array
plt.ylabel('Simulated DAQmx')
plt.xlabel('Samples')
plt.show()


KeyboardInterrupt: 